**Nombre:** Jorge Luis González Martínez

# Chatbot RAG sobre óperas de Giacomo Puccini

Se construyó un chatbot que responde preguntas sobre tres óperas de Puccini (*La bohème*, *Tosca* y *Turandot*) usando RAG (Retrieval-Augmented Generation). Como documentos se usaron los artículos de Wikipedia en español de cada ópera.

Los pasos que se siguen son: leer los documentos, limpiarlos, dividirlos en chunks, convertirlos en embeddings, buscar los chunks más parecidos a la pregunta, enviarlos a un LLM (Gemini) para que responda y, al final, evaluar qué tan bien funciona.

## 0. Preparación

El proyecto se desarrolló con Python 3.13 en un entorno virtual (`.venv`). Inicialmente se intentó con Python 3.15, pero PyTorch (necesario para los embeddings) todavía no tiene versión para él, por lo que se creó un entorno aparte. Así, las librerías del proyecto no se mezclan con las de otros proyectos.

En la siguiente celda se instalan todas las librerías utilizadas.

In [48]:
%pip install pypdf sentence-transformers google-genai pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:
import sys
print(sys.version)
print(sys.executable)  # debe apuntar a la carpeta .venv del proyecto

3.13.16 (tags/v3.13.16:cbc944f, Sep 30 2026, 19:17:50) [MSC v.1944 64 bit (AMD64)]
c:\Users\jorge\Documents\Chatbot_rag\.venv\Scripts\python.exe


## 1. Lectura de los documentos

Se leen los 3 PDFs y se cuentan las palabras de cada uno para comprobar que son lo suficientemente largos (la tarea pide como mínimo el tamaño de un paper). Los tres superan las 4,000 palabras.

In [50]:
from pathlib import Path
from pypdf import PdfReader

carpeta = Path("documentos")
textos = {}

for ruta_pdf in sorted(carpeta.glob("*.pdf")):
    lector = PdfReader(ruta_pdf)
    texto = ""
    for pagina in lector.pages:
        texto += pagina.extract_text() + "\n"
    textos[ruta_pdf.stem] = texto
    print(f"{ruta_pdf.stem}: {len(lector.pages)} páginas, {len(texto.split())} palabras")

La_bohème_(Puccini): 10 páginas, 4359 palabras
Tosca_(ópera): 11 páginas, 4282 palabras
Turandot: 12 páginas, 4772 palabras


In [51]:
nombre = list(textos.keys())[0]
print(nombre)
print(textos[nombre][:2000])

La_bohème_(Puccini)
La bohemia
La bohème
Póster original de 1896 de La bohème por Adolfo
Hohenstein.
Género Ópera verista
Actos 4 actos
Ambientada en París
Basado en Henri Murger: Scènes de la
vie de bohème (1845-49)
Publicación
Año de
publicación
siglo XIX
Idioma Italiano
Música
Compositor Giacomo Puccini
Puesta en escena
Lugar de estreno Teatro Regio (Turín)
La bohème (Puccini)
La bohème (en español, La bohemia) es una ópera
en cuatro actos3  con música de Giacomo Puccini y
libreto en italiano de Giuseppe Giacosa y Luigi
Illica, quienes simplificaron y aunaron los
diferentes episodios de la novela por entregas
Escenas de la vida bohemia, de Henri Murger,4
publicada a lo largo de cinco años (1845-1849) en
el periódico El Corsario. Refleja las experiencias
del compositor durante los años que vivió como
estudiante en el Conservatorio de Milán, donde
compartió habitación con Pietro Mascagni.5 6
El estreno de La bohème tuvo lugar en el Teatro
Regio de Turín el 1 de febrero de 1896,7  diri

Al revisar el texto extraído se observaron varios problemas: el cuadro lateral de Wikipedia se mezcla con el texto, hay saltos de línea a la mitad de las oraciones (porque el PDF está en columnas), los números de las notas al pie quedan pegados a las palabras y los títulos de las secciones quedan fuera de lugar.

## 2. Limpieza

Antes de dividir el texto se hicieron dos limpiezas:
- Se cortó cada documento donde empiezan las referencias y la bibliografía, ya que no responden ninguna pregunta y solo agregarían ruido a la búsqueda.
- Se unieron los saltos de línea y espacios repetidos en un solo espacio.

Otros detalles, como los números de las notas al pie o las tablas, no se limpiaron y quedan como mejora a futuro.

In [52]:
import re

textos_limpios = {}

for nombre, texto in textos.items():
    if "Turandot" in nombre:
        marcador = "Elisa Alberti"   # inicio de la bibliografía de Turandot
    else:
        marcador = "Triana"          # inicio de las referencias de La bohème y Tosca

    posicion = texto.find(marcador)
    if posicion == -1:
        print(f"⚠️ No encontré el marcador en {nombre}, lo dejo completo")
    else:
        texto = texto[:posicion]

    texto = re.sub(r"\s+", " ", texto).strip()
    textos_limpios[nombre] = texto
    print(f"{nombre}: {len(texto.split())} palabras después de limpiar")

La_bohème_(Puccini): 3805 palabras después de limpiar
Tosca_(ópera): 3984 palabras después de limpiar
Turandot: 3455 palabras después de limpiar


In [53]:
# Verificar que cada documento termine en texto útil y no en referencias
for nombre, texto in textos_limpios.items():
    print(nombre)
    print("...", texto[-300:])
    print()

La_bohème_(Puccini)
... rismäki estrenada en 1992. Obras derivadas Modernizaciones En el tango Griseta (letra de José González Castillo, música de Enrique Pedro Delfino, 1924) la primera estrofa hace referencia a los personajes de la ópera: Mezcla rara de Musetta y de Mimí /con caricias de Rodolfo y de Schaunard, 1. Martín

Tosca_(ópera)
... as por el barítono Robert Merrill. En la película Quantum of Solace de la serie de películas de James Bond, hay una escena clave de la misma que se desarrolla tras bastidores del Bregenz Open Air Theatre en Austria. Se ve la parte en que se hace la falsa simulación de la ejecución.13 Notas 1. Martín

Turandot
... Karajan, Orquesta y coro de la Ópera Estatal de Viena, coro de niños de Viena Audio CD: Deutsche Grammophon Cat: 423855 1984 Eva Marton, José Carreras, Katia Ricciarelli, Kurt Rydl Lorin Maazel, Orquesta Filarmónica de Viena, Coro de la Ópera Estatal de Viena, Audio CD: CBS Masterworks Cat: M2J39160



## 3. Chunking

Se revisaron varias estrategias para dividir el texto: tamaño fijo, tamaño fijo con traslape, por oraciones o párrafos, por secciones del documento y semántica.

Se eligió **tamaño fijo con traslape**: chunks de 150 palabras, con 40 palabras repetidas entre un chunk y el siguiente.
- La división por secciones no era confiable, porque al leer el PDF los títulos quedaron revueltos.
- 150 palabras equivalen aproximadamente a un párrafo largo: alcanzan para una idea completa, sin ser tan largos como para que el embedding mezcle muchos temas.
- El traslape evita que una respuesta quede cortada entre dos chunks.

Además, a cada chunk se le agregó al inicio el nombre de su ópera (por ejemplo `[Tosca] ...`). Así, aunque un fragmento no mencione de qué ópera habla, conserva ese contexto. Esto ayuda en casos como *Tosca* y *La bohème*, que tienen los mismos libretistas.

In [54]:
def partir_en_chunks(texto, nombre_doc, tam=150, traslape=40):
    palabras = texto.split()
    chunks = []
    paso = tam - traslape

    for inicio in range(0, len(palabras), paso):
        pedazo = palabras[inicio:inicio + tam]
        chunks.append({
            "id": f"{nombre_doc}_{len(chunks)}",
            "documento": nombre_doc,
            "texto": f"[{nombre_doc}] " + " ".join(pedazo),
        })
        if inicio + tam >= len(palabras):
            break

    return chunks

In [55]:
todos_los_chunks = []

for nombre, texto in textos_limpios.items():
    if "Tosca" in nombre:
        titulo = "Tosca"
    elif "Turandot" in nombre:
        titulo = "Turandot"
    else:
        titulo = "La bohème"

    chunks_doc = partir_en_chunks(texto, titulo)
    todos_los_chunks.extend(chunks_doc)
    print(f"{titulo}: {len(chunks_doc)} chunks")

print("Total de chunks:", len(todos_los_chunks))

La bohème: 35 chunks
Tosca: 36 chunks
Turandot: 32 chunks
Total de chunks: 103


In [56]:
# Ejemplo: los dos primeros chunks (primeros 300 caracteres)
for c in todos_los_chunks[:2]:
    print(c["id"])
    print(c["texto"][:300], "...")
    print()

La bohème_0
[La bohème] La bohemia La bohème Póster original de 1896 de La bohème por Adolfo Hohenstein. Género Ópera verista Actos 4 actos Ambientada en París Basado en Henri Murger: Scènes de la vie de bohème (1845-49) Publicación Año de publicación siglo XIX Idioma Italiano Música Compositor Giacomo Puccini  ...

La bohème_1
[La bohème] cinco años (1845-1849) en el periódico El Corsario. Refleja las experiencias del compositor durante los años que vivió como estudiante en el Conservatorio de Milán, donde compartió habitación con Pietro Mascagni.5 6 El estreno de La bohème tuvo lugar en el Teatro Regio de Turín el 1 de f ...



In [57]:
# Comprobar el traslape: las últimas 40 palabras del chunk 0 deben ser
# las primeras 40 del chunk 1 (después de la etiqueta de 2 palabras "[La bohème]")
palabras_chunk0 = todos_los_chunks[0]["texto"].split()
palabras_chunk1 = todos_los_chunks[1]["texto"].split()

fin_chunk0 = palabras_chunk0[-40:]
inicio_chunk1 = palabras_chunk1[2:42]

print("¿Son iguales?", fin_chunk0 == inicio_chunk1)

¿Son iguales? True


## 4. Embeddings

Un embedding convierte un texto en un vector de números, de forma que los textos con significado parecido quedan cerca entre sí. Esto permite buscar por significado y no solo por palabras iguales.

Se usó el modelo `intfloat/multilingual-e5-small` (vectores de 384 números) porque:
- Es multilingüe y funciona bien en español.
- Es gratuito y corre de forma local.
- Acepta textos de hasta 512 tokens. Otros modelos populares solo leen 128 tokens (unas 90 palabras) y habrían cortado la mitad de cada chunk sin avisar.

Este modelo requiere que las preguntas empiecen con `query: ` y los fragmentos con `passage: `, ya que así fue entrenado.

In [58]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("intfloat/multilingual-e5-small")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2434.10it/s]


In [59]:
# Experimento: una pregunta contra tres pasajes
frases = [
    "query: ¿Cuándo se estrenó Tosca?",
    "passage: Tosca fue estrenada en Roma el 14 de enero de 1900.",
    "passage: Turandot quedó inconclusa por la muerte de Puccini.",
    "passage: El pan se hace con harina, agua y levadura.",
]

vectores = modelo.encode(frases, normalize_embeddings=True)

print("Forma:", vectores.shape)
print("Primeros 5 números de la pregunta:", vectores[0][:5])
print()

similitudes = vectores[0] @ vectores[1:].T
for frase, sim in zip(frases[1:], similitudes):
    print(f"{sim:.3f}  {frase}")

Forma: (4, 384)
Primeros 5 números de la pregunta: [ 0.05181877 -0.02276542 -0.065699   -0.08243337  0.08890343]

0.904  passage: Tosca fue estrenada en Roma el 14 de enero de 1900.
0.747  passage: Turandot quedó inconclusa por la muerte de Puccini.
0.717  passage: El pan se hace con harina, agua y levadura.


El pasaje que responde la pregunta obtiene la similitud más alta. Con este modelo los valores suelen ser todos altos (0.7–0.9): lo que importa es **el orden**, no el valor absoluto.

In [60]:
textos_chunks = ["passage: " + c["texto"] for c in todos_los_chunks]

embeddings_chunks = modelo.encode(textos_chunks, normalize_embeddings=True)

print("Forma:", embeddings_chunks.shape)

Forma: (103, 384)


## 5. Recuperación

Para buscar, la pregunta se convierte en embedding y se compara con los 103 chunks usando similitud coseno (como los vectores están normalizados, equivale a un producto punto). Se conservan los k chunks más parecidos; es la misma idea que KNN. En la versión final se usó k = 5 (en la sección 9 se explica por qué).

Como son solo 103 chunks, basta con guardarlos en memoria. Con una cantidad mucho mayor de documentos se usaría una base de datos vectorial como FAISS o Chroma.

In [61]:
def buscar(pregunta, k=3):
    emb_pregunta = modelo.encode("query: " + pregunta, normalize_embeddings=True)
    similitudes = embeddings_chunks @ emb_pregunta
    mejores = similitudes.argsort()[::-1][:k]

    resultados = []
    for i in mejores:
        resultados.append({
            "id": todos_los_chunks[i]["id"],
            "similitud": float(similitudes[i]),
            "texto": todos_los_chunks[i]["texto"],
        })
    return resultados

In [62]:
# Pregunta "trampa": Tosca y La bohème tienen los mismos libretistas
pregunta = "¿Quién escribió el libreto de Tosca?"

for r in buscar(pregunta):
    print(r["id"], "| similitud:", round(r["similitud"], 3))
    print(r["texto"][:300])
    print()

Tosca_0 | similitud: 0.881
[Tosca] Tosca Tosca Cartel original. Género Ópera verista Actos Tres actos Ambientada en Roma Basado en Victorien Sardou: La Tosca (drama) Publicación Año de publicación 1900 Idioma Italiano Música Compositor Giacomo Puccini Puesta en escena Lugar de estreno Teatro Costanzi (Roma) Fecha de estreno 1

Tosca_4 | similitud: 0.863
[Tosca] a Puccini hacerlo. Si bien inicialmente Puccini se mostró ofendido, Verdi intercedió para convencerlo. Puccini comenzó a componer la ópera en 1896, tras haber terminado La bohème. Ricordi encomendó a Giacosa trabajar junto a Luigi Illica en el libreto. En octubre de 1899, luego de tres años 

Tosca_3 | similitud: 0.861
[Tosca] revolucionario (bajo) el sacristán (barítono) Spoletta, policía (tenor buffo) Sciarrone, gendarme (bajo) Un carcelero (bajo) Un pastor (voz blanca) Nobles, religiosos, soldados1 Libretista G. Giacosa, L. Illica Duración 2 horas Puccini pidió inmediatamente a su editor, Giulio Ricordi, que ad



## 6. Generación con el LLM

Como LLM se usó Google Gemini con el plan gratuito. Se tomaron las siguientes decisiones:
- **API key:** se solicita con `getpass` para que no quede escrita en el notebook.
- **Tiempo límite:** si Gemini no responde en 60 segundos, la petición se considera fallida, para que el notebook no se quede detenido.
- **Reintentos:** si el servicio está saturado (error 503) o se alcanza el límite de uso (error 429), se vuelve a intentar esperando 10, 20 y 40 segundos.
- **Versión fija del modelo:** se usa una versión concreta y no un alias como `latest`, que puede cambiar sin aviso y alterar los resultados.
- **Modelo:** primero se probó `gemini-3.8-flash`, pero estaba saturado; después se usó `gemini-3.5-flash` hasta agotar la cuota gratuita, por lo que la versión final usa `gemini-3.5-flash-lite`. Para leer unos fragmentos y dar una respuesta corta, un modelo ligero es suficiente.

In [63]:
from getpass import getpass
from google import genai
from google.genai import types

api_key = getpass("Pega tu API key de Gemini y presiona Enter: ")
cliente = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(timeout=60_000),  # 60 segundos
)

In [64]:
# Modelos "flash" disponibles para esta cuenta
for m in cliente.models.list():
    if "flash" in m.name:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [65]:
import time

MODELO = "gemini-3.5-flash-lite"

def preguntar_llm(prompt, intentos=4, espera=10):
    for intento in range(1, intentos + 1):
        try:
            respuesta = cliente.models.generate_content(model=MODELO, contents=prompt)
            return respuesta.text
        except Exception as error:
            print(f"Intento {intento} falló: {str(error)[:120]}")
            if intento < intentos:
                print(f"Esperando {espera} segundos antes de reintentar...")
                time.sleep(espera)
                espera = espera * 2

    raise RuntimeError("Gemini no respondió después de varios intentos.")

In [66]:
print(preguntar_llm("Saluda en una frase corta y dime qué modelo eres."))

¡Hola! Soy Gemini, un modelo de inteligencia artificial desarrollado por Google.


**Ejemplo de alucinación:** se llamó a `gemini-3.5-flash`, pero el modelo afirma ser "Gemini 3.6 Flash". Sin contexto, un LLM puede responder con seguridad algo falso. Este es el problema que RAG busca mitigar.

## 7. Chatbot RAG

El chatbot integra todo en tres pasos:
1. **Recuperar** los chunks más parecidos a la pregunta.
2. **Aumentar** la pregunta: se arma un prompt con instrucciones, los chunks y la pregunta.
3. **Generar** la respuesta con Gemini.

La instrucción más importante del prompt es que responda solo con la información del contexto y que, si no la encuentra, lo indique en lugar de inventar.

In [67]:
def armar_prompt(pregunta, chunks):
    contexto = "\n\n".join(f"[Fragmento {c['id']}]\n{c['texto']}" for c in chunks)

    return f"""Eres un asistente experto en las óperas de Giacomo Puccini.
Responde la pregunta usando ÚNICAMENTE la información del contexto.
Si la respuesta no está en el contexto, responde exactamente: "No encontré esa información en los documentos."
Responde en español, de forma breve y directa.

CONTEXTO:
{contexto}

PREGUNTA: {pregunta}

RESPUESTA:"""

In [68]:
def chatbot_rag(pregunta, k=5):
    chunks = buscar(pregunta, k=k)
    prompt = armar_prompt(pregunta, chunks)
    respuesta = preguntar_llm(prompt)

    return {
        "respuesta": respuesta,
        "chunks": [c["id"] for c in chunks],
    }

### Pruebas

Se probó el chatbot con 4 preguntas: dos cuya respuesta sí está en los documentos y dos que no. Una de estas últimas es el año de estreno de *Madama Butterfly*, que Gemini sí conoce de memoria, para comprobar si respeta la regla de usar solo el contexto. Se dejan unos segundos entre preguntas para no rebasar el límite de consultas por minuto.

In [69]:
pruebas = [
    "¿Quién terminó de componer Turandot después de la muerte de Puccini?",  # respuesta en los documentos
    "¿En qué año se estrenó Madama Butterfly?",  # NO está en los documentos (Gemini sí lo sabe de memoria)
    "¿Quién cantó a Mario Cavaradossi en la Tosca de la Orquesta Filarmónica de Toluca?",  # NO está en los documentos
    "¿Qué tipo de voz se necesita para cantar el papel de Floria Tosca?",  # respuesta en los documentos
]

for pregunta in pruebas:
    resultado = chatbot_rag(pregunta)
    print("Pregunta:", pregunta)
    print("Respuesta:", resultado["respuesta"])
    print("Chunks usados:", resultado["chunks"])
    print()
    time.sleep(5)

Pregunta: ¿Quién terminó de componer Turandot después de la muerte de Puccini?
Respuesta: Franco Alfano
Chunks usados: ['Turandot_5', 'Turandot_6', 'Turandot_0', 'Turandot_7', 'Turandot_8']

Pregunta: ¿En qué año se estrenó Madama Butterfly?
Respuesta: No encontré esa información en los documentos.
Chunks usados: ['Turandot_8', 'La bohème_9', 'La bohème_7', 'Tosca_1', 'La bohème_6']

Pregunta: ¿Quién cantó a Mario Cavaradossi en la Tosca de la Orquesta Filarmónica de Toluca?
Respuesta: No encontré esa información en los documentos.
Chunks usados: ['Tosca_9', 'Tosca_27', 'Tosca_18', 'Tosca_15', 'Tosca_7']

Intento 1 falló: The read operation timed out
Esperando 10 segundos antes de reintentar...
Pregunta: ¿Qué tipo de voz se necesita para cantar el papel de Floria Tosca?
Respuesta: Soprano (específicamente soprano lírica y spinto).
Chunks usados: ['Tosca_9', 'Tosca_26', 'Tosca_7', 'Tosca_27', 'Tosca_0']



## 8. Conjunto de evaluación

Se escribieron 20 preguntas con su respuesta correcta (en `preguntas_evaluacion.csv`), repartidas entre las tres óperas. Algunas se incluyeron a propósito por ser difíciles, por ejemplo, preguntas donde dos óperas tienen datos parecidos.

Para evaluar la búsqueda es necesario saber en qué chunks está cada respuesta. Para encontrarlos se usó una búsqueda de texto exacta con palabras clave, y no el buscador de embeddings, ya que si se usara el mismo buscador que se va a evaluar, se estaría calificando a sí mismo. Después, los candidatos se revisaron manualmente.

In [70]:
import pandas as pd

preguntas = pd.read_csv("preguntas_evaluacion.csv")
preguntas.head()

,id,opera,pregunta,respuesta_esperada,trampa,chunks_relevantes
0,1,La bohème,¿Qué tipo de voz se necesita para cantar el pa...,Soprano lírica,No,La bohème_2;La bohème_3
1,2,La bohème,¿Qué tipo de voz se necesita para cantar el pa...,Tenor lírico o lírico-spinto,No,La bohème_2;La bohème_3
2,3,La bohème,¿Cuál es el motivo del fallecimiento de Mimí?,"Una enfermedad mortal, muy probablemente tuber...",No,La bohème_23
3,4,La bohème,¿En dónde se ambienta la ópera La Boheme?,"En París (Barrio Latino), alrededor de 1830",No,La bohème_0;La bohème_14;La bohème_15
4,5,La bohème,"¿En qué acto de La Bohème se canta el dueto ""O...",En el Acto I,Sí,La bohème_19


In [71]:
claves = {
    1: ["soprano lírica"],
    2: ["lírico-spinto"],
    3: ["tuberculosis"],
    4: ["Ambientada en París", "1830"],
    5: ["soave fanciulla"],
    6: ["gelida manina"],
    7: ["Pini-Corsi"],
    8: ["supuesta derrota"],
    9: ["Spoletta, policía", "Spoletta, agente"],
    10: ["lucevan"],
    11: ["Andrea della Valle"],
    12: ["segunda mitad del siglo"],
    13: ["componer la ópera en 1896"],
    14: ["número 6"],
    15: ["Nessun dorma"],
    16: ["hijo de Timur", "figlio di Timur"],
    17: ["la esperanza"],
    18: ["Toscanini"],
    19: ["Ambientada en Pekín", "Pekín"],
    20: ["sobreagudos"],
}

In [72]:
def candidatos(id_pregunta):
    opera = preguntas.loc[preguntas["id"] == id_pregunta, "opera"].iloc[0]
    encontrados = []
    for c in todos_los_chunks:
        if c["documento"] != opera:
            continue
        for palabra in claves[id_pregunta]:
            if palabra.lower() in c["texto"].lower():
                encontrados.append(c["id"])
                break
    return encontrados

def ver(id_chunk):
    for c in todos_los_chunks:
        if c["id"] == id_chunk:
            print(c["texto"])

for _, fila in preguntas.iterrows():
    print(fila["id"], fila["pregunta"])
    print("   ", candidatos(fila["id"]))

1 ¿Qué tipo de voz se necesita para cantar el papel de Mimí de La Bohème?
    ['La bohème_2', 'La bohème_3']
2 ¿Qué tipo de voz se necesita para cantar el papel de Rodolfo de La Bohème?
    ['La bohème_2', 'La bohème_3']
3 ¿Cuál es el motivo del fallecimiento de Mimí?
    ['La bohème_23']
4 ¿En dónde se ambienta la ópera La Boheme?
    ['La bohème_0', 'La bohème_14', 'La bohème_15', 'La bohème_31', 'La bohème_32']
5 ¿En qué acto de La Bohème se canta el dueto "O soave fanciulla"?
    ['La bohème_19']
6 ¿Qué aria canta Rodolfo cuando conoce a Mimí?
    ['La bohème_18']
7 ¿Quién cantó el papel de Schaunard en el estreno de La bohème?
    ['La bohème_14']
8 ¿Qué se celebra con el Te Deum al final del primer acto de Tosca?
    ['Tosca_7']
9 ¿Qué tipo de voz se necesita para cantar el personaje de Spoletta?
    ['Tosca_2', 'Tosca_3', 'Tosca_7', 'Tosca_8']
10 ¿Cuál es el aria más famosa que canta Mario Cavaradossi?
    ['Tosca_1', 'Tosca_2', 'Tosca_25']
11 ¿En qué iglesia se refugia Angelott

### Revisión manual

Con `ver("id_del_chunk")` se leyeron los candidatos dudosos. De 43 candidatos, se descartaron 11 que contenían la palabra clave pero no respondían la pregunta. Por ejemplo, en la pregunta 4, "1830" también aparece en la versión de Baz Luhrmann ambientada en 1957, y en la 19, "Pekín" aparece en varias escenas, pero solo un chunk indica dónde se ambienta la ópera.

También se agregaron 2 chunks a la pregunta 16 que las palabras clave no habían encontrado; esto se detectó al analizar los fallos de la sección 9.

In [73]:
ver("La bohème_31")

[La bohème] 1959 el vals de Musetta fue adaptado por el escritor de canciones Bobby Worth para la canción pop del año 1959 Don't You Know?, un éxito de Della Reese.17 La ópera fue también adaptada en forma de cuento en 1983 por el novelista V. S. Pritchett para ser publicado por la Asociación de la Metropolitan Opera.18 La Bohème. Una piccola storia sull'immortalità dell'amore e dell'amicizia por Carollina Fabinger es una versión ilustrada en italiano para jóvenes lectores publicado por Nuages, Milán 2009, ISBN 978-88- 86178-89-1. Baz Luhrmann produjo la ópera para Opera Australia en 199019 con traducciones subtituladas modernizadas y un presupuesto de solo 60 000 AUD. Se lanzó un DVD con la representación en el escenario. Esta versión se ambientaba en 1957, más que en el período original de 1830.19 La razón para actualizar La bohème a esta época, según Baz Luhrmann, fue que «... (ellos) descubrieron que 1957


In [74]:
# Se parte de los candidatos automáticos y luego se corrigen a mano
relevantes = {int(i): candidatos(i) for i in preguntas["id"]}

# Correcciones tras la revisión manual (descomenta y ajusta). Ejemplo:
# relevantes[4] = ["La bohème_0", "La bohème_14", "La bohème_15"]
relevantes[4] = ["La bohème_0", "La bohème_14", "La bohème_15"]
relevantes[10] = ["Tosca_25"]
relevantes[11] = ["Tosca_8"]
relevantes[15] = ["Turandot_1", "Turandot_22"]
relevantes[16] = ["Turandot_13", "Turandot_14", "Turandot_26"]
relevantes[18] = ["Turandot_6", "Turandot_11"]
relevantes[19] = ["Turandot_0"]


for i, ids in relevantes.items():
    print(i, ids)

1 ['La bohème_2', 'La bohème_3']
2 ['La bohème_2', 'La bohème_3']
3 ['La bohème_23']
4 ['La bohème_0', 'La bohème_14', 'La bohème_15']
5 ['La bohème_19']
6 ['La bohème_18']
7 ['La bohème_14']
8 ['Tosca_7']
9 ['Tosca_2', 'Tosca_3', 'Tosca_7', 'Tosca_8']
10 ['Tosca_25']
11 ['Tosca_8']
12 ['Tosca_26', 'Tosca_27']
13 ['Tosca_3', 'Tosca_4']
14 ['Tosca_4']
15 ['Turandot_1', 'Turandot_22']
16 ['Turandot_13', 'Turandot_14', 'Turandot_26']
17 ['Turandot_20']
18 ['Turandot_6', 'Turandot_11']
19 ['Turandot_0']
20 ['Turandot_0', 'Turandot_1']


In [75]:
# ⚠️ Correr SOLO después de revisar: guarda los chunks relevantes en el CSV
preguntas["chunks_relevantes"] = preguntas["id"].map(lambda i: ";".join(relevantes[i]))
preguntas.to_csv("preguntas_evaluacion.csv", index=False, encoding="utf-8-sig")
preguntas[["id", "chunks_relevantes"]]

,id,chunks_relevantes
0,1,La bohème_2;La bohème_3
1,2,La bohème_2;La bohème_3
2,3,La bohème_23
3,4,La bohème_0;La bohème_14;La bohème_15
4,5,La bohème_19
5,6,La bohème_18
6,7,La bohème_14
7,8,Tosca_7
8,9,Tosca_2;Tosca_3;Tosca_7;Tosca_8
9,10,Tosca_25


## 9. Evaluación de la recuperación

Para medir si el buscador trae el chunk correcto se usaron dos métricas:
- **Hit Rate@k:** porcentaje de preguntas en las que aparece al menos un chunk correcto entre los k recuperados. Se eligió porque el LLM solo ve esos k chunks: si el correcto no está ahí, es imposible que responda bien.
- **MRR (Mean Reciprocal Rank):** considera la posición en la que aparece el primer chunk correcto (1.º vale 1, 2.º vale 0.5, 3.º vale 0.33). Complementa al Hit Rate porque también premia que el chunk correcto aparezca más arriba.

In [76]:
def evaluar_recuperacion(k=3):
    filas = []
    for _, fila in preguntas.iterrows():
        correctos = relevantes[fila["id"]]
        recuperados = [r["id"] for r in buscar(fila["pregunta"], k=k)]

        hit = 1 if any(c in correctos for c in recuperados) else 0

        rr = 0
        for posicion, chunk_id in enumerate(recuperados, start=1):
            if chunk_id in correctos:
                rr = 1 / posicion
                break

        filas.append({
            "id": fila["id"],
            "pregunta": fila["pregunta"],
            "recuperados": recuperados,
            "hit": hit,
            "rr": rr,
        })
    return pd.DataFrame(filas)

res_recuperacion = evaluar_recuperacion(k=3)

print("Hit Rate@3:", res_recuperacion["hit"].mean())
print("MRR@3:", round(res_recuperacion["rr"].mean(), 3))

Hit Rate@3: 0.75
MRR@3: 0.608


Preguntas en las que el buscador no recuperó ningún chunk correcto con k = 3.

In [77]:
res_recuperacion[res_recuperacion["hit"] == 0]

,id,pregunta,recuperados,hit,rr
0,1,¿Qué tipo de voz se necesita para cantar el pa...,"[La bohème_28, La bohème_32, La bohème_14]",0,0.0
2,3,¿Cuál es el motivo del fallecimiento de Mimí?,"[La bohème_27, La bohème_14, La bohème_28]",0,0.0
3,4,¿En dónde se ambienta la ópera La Boheme?,"[La bohème_10, La bohème_9, La bohème_11]",0,0.0
6,7,¿Quién cantó el papel de Schaunard en el estre...,"[La bohème_33, La bohème_32, La bohème_16]",0,0.0
7,8,¿Qué se celebra con el Te Deum al final del pr...,"[Tosca_22, Tosca_1, Tosca_24]",0,0.0


### Comparación con distintos valores de k

Se compararon k = 1, 3 y 5. Con k = 5 el Hit Rate sube a 0.90, mientras que el MRR casi no cambia; esto indica que en varias preguntas el chunk correcto sí aparece, pero en la posición 4 o 5. Como 5 chunks son unas 750 palabras (muy poco para Gemini), se decidió usar k = 5 en el chatbot.

In [78]:
for k in [1, 3, 5]:
    r = evaluar_recuperacion(k=k)
    print(f"k={k}:  Hit Rate = {r['hit'].mean():.2f}   MRR = {r['rr'].mean():.3f}")

k=1:  Hit Rate = 0.50   MRR = 0.500
k=3:  Hit Rate = 0.75   MRR = 0.608
k=5:  Hit Rate = 0.90   MRR = 0.643


## 10. Evaluación de la generación

Primero, el chatbot responde las 20 preguntas con k = 5. Las respuestas se guardan en `respuestas_chatbot.csv` después de cada pregunta, para no perder el avance si se agota la cuota de Gemini.

Para calificar las respuestas se usaron dos métricas:
- **Similitud semántica:** se compara, mediante embeddings, la respuesta del chatbot con la respuesta correcta. Es automática y no consume cuota, pero solo mide si los textos se parecen.
- **LLM como juez:** se le pide a Gemini que califique cada respuesta como CORRECTA, PARCIAL o INCORRECTA, comparándola con la respuesta correcta. Se eligió porque las respuestas son texto libre ("Franco Alfano" y "Fue completada por Franco Alfano" son ambas correctas) y una comparación exacta no funcionaría. Posteriormente, todas sus calificaciones se revisaron manualmente.

In [79]:
import os

K_FINAL = 5
ARCHIVO_RESP = "respuestas_chatbot.csv"

if os.path.exists(ARCHIVO_RESP):
    respuestas = pd.read_csv(ARCHIVO_RESP)
else:
    respuestas = pd.DataFrame(columns=["id", "respuesta_chatbot", "chunks_usados"])

hechas = set(respuestas["id"])

for _, fila in preguntas.iterrows():
    if fila["id"] in hechas:
        continue

    resultado = chatbot_rag(fila["pregunta"], k=K_FINAL)

    nueva = {
        "id": fila["id"],
        "respuesta_chatbot": resultado["respuesta"],
        "chunks_usados": ";".join(resultado["chunks"]),
    }
    respuestas = pd.concat([respuestas, pd.DataFrame([nueva])], ignore_index=True)
    respuestas.to_csv(ARCHIVO_RESP, index=False, encoding="utf-8-sig")

    print(fila["id"], "→", resultado["respuesta"][:80])
    time.sleep(6)

print("Respuestas guardadas:", len(respuestas), "de", len(preguntas))

Respuestas guardadas: 20 de 20


Se unen en una sola tabla las preguntas, sus respuestas correctas y las respuestas del chatbot.

In [80]:
respuestas = pd.read_csv(ARCHIVO_RESP)
datos = preguntas.merge(respuestas, on="id")
datos[["id", "respuesta_esperada", "respuesta_chatbot"]]

,id,respuesta_esperada,respuesta_chatbot
0,1,Soprano lírica,Soprano
1,2,Tenor lírico o lírico-spinto,Tenor lírico o lírico-spinto.
2,3,"Una enfermedad mortal, muy probablemente tuber...",No encontré esa información en los documentos.
3,4,"En París (Barrio Latino), alrededor de 1830","La ópera *La bohème* se ambienta en París, alr..."
4,5,En el Acto I,No encontré esa información en los documentos.
5,6,Che gelida manina («Qué manita más fría»),Rodolfo interpreta el aria *Che gelida manina*...
6,7,Antonio Pini-Corsi,No encontré esa información en los documentos.
7,8,La supuesta derrota de Napoleón,La supuesta derrota de Napoleón.
8,9,Tenor buffo,Tenor buffo
9,10,E lucevan le stelle,El aria *E lucevan le stelle*.


In [81]:
emb_esperada = modelo.encode(["query: " + t for t in datos["respuesta_esperada"]], normalize_embeddings=True)
emb_chatbot = modelo.encode(["query: " + str(t) for t in datos["respuesta_chatbot"]], normalize_embeddings=True)

datos["similitud"] = (emb_esperada * emb_chatbot).sum(axis=1)
print("Similitud semántica promedio:", round(datos["similitud"].mean(), 3))

Similitud semántica promedio: 0.908


In [82]:
def armar_prompt_juez(pregunta, esperada, chatbot):
    return f"""Eres un evaluador estricto. Compara la respuesta de un chatbot con la respuesta correcta.

PREGUNTA: {pregunta}
RESPUESTA CORRECTA: {esperada}
RESPUESTA DEL CHATBOT: {chatbot}

En la primera línea escribe UNA sola palabra:
CORRECTA: contiene la información esencial de la respuesta correcta, aunque use otras palabras.
PARCIAL: contiene solo una parte de la información esencial.
INCORRECTA: es errónea, contradice la respuesta correcta o no responde.
En la segunda línea explica en una frase por qué."""

def leer_veredicto(texto):
    texto = texto.upper()
    if "INCORRECTA" in texto:
        return "INCORRECTA"
    if "PARCIAL" in texto:
        return "PARCIAL"
    if "CORRECTA" in texto:
        return "CORRECTA"
    return "SIN VEREDICTO"

In [83]:
ARCHIVO_JUEZ = "calificaciones_juez.csv"

if os.path.exists(ARCHIVO_JUEZ):
    juez = pd.read_csv(ARCHIVO_JUEZ)
else:
    juez = pd.DataFrame(columns=["id", "veredicto", "explicacion"])

hechas = set(juez["id"])

for _, fila in datos.iterrows():
    if fila["id"] in hechas:
        continue

    texto = preguntar_llm(armar_prompt_juez(fila["pregunta"], fila["respuesta_esperada"], fila["respuesta_chatbot"]))

    nueva = {"id": fila["id"], "veredicto": leer_veredicto(texto), "explicacion": texto.strip()}
    juez = pd.concat([juez, pd.DataFrame([nueva])], ignore_index=True)
    juez.to_csv(ARCHIVO_JUEZ, index=False, encoding="utf-8-sig")

    print(fila["id"], "→", nueva["veredicto"])
    time.sleep(6)

In [84]:
juez = pd.read_csv(ARCHIVO_JUEZ)
datos = datos.drop(columns=["veredicto", "explicacion"], errors="ignore").merge(juez, on="id")

puntos = {"CORRECTA": 1, "PARCIAL": 0.5, "INCORRECTA": 0}
datos["puntaje"] = datos["veredicto"].map(puntos)
datos["abstuvo"] = datos["respuesta_chatbot"].str.contains("No encontré")

print("Exactitud (LLM juez):", round(datos["puntaje"].mean(), 3))
print(datos["veredicto"].value_counts())
print("Respuestas 'No encontré':", datos["abstuvo"].sum())
print("Similitud semántica promedio:", round(datos["similitud"].mean(), 3))

datos[["id", "respuesta_chatbot", "veredicto", "similitud"]]

Exactitud (LLM juez): 0.825
veredicto
CORRECTA      16
INCORRECTA     3
PARCIAL        1
Name: count, dtype: int64
Respuestas 'No encontré': 3
Similitud semántica promedio: 0.908


,id,respuesta_chatbot,veredicto,similitud
0,1,Soprano,PARCIAL,0.916365
1,2,Tenor lírico o lírico-spinto.,CORRECTA,0.988006
2,3,No encontré esa información en los documentos.,INCORRECTA,0.792302
3,4,"La ópera *La bohème* se ambienta en París, alr...",CORRECTA,0.895908
4,5,No encontré esa información en los documentos.,INCORRECTA,0.816875
5,6,Rodolfo interpreta el aria *Che gelida manina*...,CORRECTA,0.935039
6,7,No encontré esa información en los documentos.,INCORRECTA,0.757999
7,8,La supuesta derrota de Napoleón.,CORRECTA,0.987804
8,9,Tenor buffo,CORRECTA,1.000000
9,10,El aria *E lucevan le stelle*.,CORRECTA,0.944440


Por último, se cruzaron los resultados de la búsqueda (k = 5) con las calificaciones del juez, para identificar si los errores provienen de la búsqueda o de la generación.

In [85]:
res5 = evaluar_recuperacion(k=5)[["id", "hit"]]
diagnostico = datos.merge(res5, on="id")

pd.crosstab(diagnostico["hit"], diagnostico["veredicto"])

veredicto,CORRECTA,INCORRECTA,PARCIAL
hit,,,
0,0,1,1
1,16,2,0


## 11. Conclusiones
En la evaluación de desempeño, el chatbot respondió correctamente 16 de 20 preguntas, alcanzando una exactitud de 0.825 según la valoración del LLM juez, resultado que fue verificado al 100% mediante revisión manual. El aspecto más relevante del sistema fue garantizar la ausencia total de alucinaciones: cuando la información no estuvo disponible en las fuentes, el modelo indicó expresamente que no la encontró en los documentos, cumpliendo así el objetivo fundamental de la arquitectura RAG al responder únicamente con base en el contexto proporcionado y no de memoria.En cuanto a la etapa de recuperación, la configuración de $k = 5$ fragmentos permitió al buscador localizar al menos un fragmento correcto en el 90% de los casos (Hit Rate@5 = 0.90 y MRR = 0.643), superando el 75% obtenido al evaluar con solo 3 fragmentos. Por su parte, el módulo de generación demostró alta precisión, acertando en 16 de las 18 preguntas (89%) en las que la búsqueda recuperó el fragmento adecuado.Los cuatro errores identificados en el proceso tuvieron causas bien delimitadas: la mitad ocurrió porque el buscador no recuperó el fragmento correcto y la otra mitad porque la información del fragmento recuperado estaba incompleta. Los factores principales de estas fallas fueron la alteración del formato de las tablas al convertir el PDF a texto (afectando la lectura de personajes y cantantes), la separación entre los títulos y su contenido (como en el caso de "Acto I", donde el texto pierde el contexto del acto al que pertenece) y la presencia de fragmentos semánticamente parecidos pero conceptualmente distintos (por ejemplo, confundir el lugar donde se ambienta la ópera con los teatros donde fue presentada).Finalmente, el análisis evidenció que la similitud semántica por sí sola no es un indicador confiable de exactitud, dado que una respuesta de reserva como "No encontré esa información" obtuvo un puntaje de similitud de 0.817, prácticamente idéntico al 0.825 de una respuesta correcta. Esta limitación justificó el uso del LLM como juez respaldado por la revisión manual. En conclusión, el chatbot resulta altamente confiable al no inventar información y presenta un diagnóstico claro para su posterior optimización mediante la mejora en la extracción desde el PDF y el ajuste de los algoritmos de búsqueda.